<a href="https://colab.research.google.com/github/Exper626/irpWhiteTea/blob/main/models_sod.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

from google.colab import drive
drive.mount('/content/drive')

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install ultralytics -q
!pip install torch torchvision -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.4 MB/s eta 0:00:00


In [3]:
from ultralytics import YOLO
import torch
print(f'GPU available: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
GPU available: True
Device: Tesla T4


Config

In [7]:
# /content/drive/MyDrive/augment_herman/data.yaml
DATA_YAML   = '/content/drive/MyDrive/augment_herman/data.yaml'
PROJECT_DIR = '/content/drive/MyDrive/yolo_runs'
EPOCHS      = 60
IMG_SIZE    = 640
BATCH       = 16
PATIENCE    = 20   # early stopping in 20 epochs (look into this aw)

In [5]:
# Each model is selected based on a specific theoretical motivation
# Based on research done in the domain and inspiration from them
# Draft LR that was duly submitted
# Also Ultralytics documentation via Discord

MODELS = {

    # BASELINE — mirrors Wang et al. (2024) tea bud detection methodology
    # Anchor-based, CSP backbone, PANet neck
    # Establishes direct comparison with closest prior work
    'yolov5nu_whitetea': 'yolov5nu.pt',

    # ANCHOR-FREE IMPROVEMENT — decoupled head, C2f backbone
    # Wang & Zhao (2025) SOD-YOLO built on YOLOv8 for small object detection
    # Tests whether anchor-free design improves white tea bud localisation
    'yolov8n_whitetea': 'yolov8n.pt',

    # ── NMS-FREE EFFICIENCY ─────────────────────────────────────────────────
    # YOLOv10n: dual-label assignment, NMS-free end-to-end inference
    # Ultralytics documentation highlights v10 for lightweight deployment
    # and edge device suitability — relevant for field deployment context
    # Removes post-processing latency bottleneck; good for dense small objects
    # where standard NMS suppresses true positives (confirmed via Ultralytics docs)
    'yolov10n_whitetea': 'yolov10n.pt',

    # HYPOTHESIS-DRIVEN — C3k2 blocks + C2PSA spatial attention
    # Sapkota & Karkee (2025): explicitly designed for small cluttered scenes
    # Primary candidate based on literature — expected best performer
    'yolo11n_whitetea': 'yolo11n.pt',

}

Training all YOLO variants

In [8]:
results_summary = {}

for run_name, weights in MODELS.items():
    print(f'\n{"="*60}')
    print(f'Training: {run_name}')
    print(f'Weights:  {weights}')
    print(f'{"="*60}\n')

    model = YOLO(weights)

    model.train(
        data      = DATA_YAML,
        epochs    = EPOCHS,
        imgsz     = IMG_SIZE,
        batch     = BATCH,
        name      = run_name,
        project   = PROJECT_DIR,
        patience  = PATIENCE,
        save      = True,
        plots     = True,
        exist_ok  = True,
        verbose   = True
    )

    # Validate immediately after training
    metrics = model.val(data=DATA_YAML)

    results_summary[run_name] = {
        'mAP50':    round(metrics.box.map50, 4),
        'mAP50-95': round(metrics.box.map,   4),
        'Precision': round(metrics.box.mp,   4),
        'Recall':    round(metrics.box.mr,   4),
    }

    print(f'\nResults for {run_name}:')
    for k, v in results_summary[run_name].items():
        print(f'  {k}: {v}')

print('\n\nAll YOLO training complete.')


Training: yolov5nu_whitetea
Weights:  yolov5nu.pt

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/augment_herman/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov5nu.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov5nu_whitetea, nbs=64, nms=False, opset=

Comparison

In [9]:
print(f'\n{"Model":<30} {"mAP50":>8} {"mAP50-95":>10} {"Precision":>10} {"Recall":>8}')
print('-' * 70)
for name, r in results_summary.items():
    print(f'{name:<30} {r["mAP50"]:>8} {r["mAP50-95"]:>10} {r["Precision"]:>10} {r["Recall"]:>8}')


Model                             mAP50   mAP50-95  Precision   Recall
----------------------------------------------------------------------
yolov5nu_whitetea                0.1307     0.0524     0.8647   0.1071
yolov8n_whitetea                 0.2292      0.086     0.4288     0.25
yolov10n_whitetea                0.2144     0.0796     0.3455     0.25
yolo11n_whitetea                 0.2047     0.0997     0.3736   0.2143


RT-DETR

In [10]:
# RT-DETR — Transformer-based alternative architecture
# TESTING:
# Zhao, H. et al. (2023) ‘DETRs beat YOLOs on real-time object detection’
# Tests whether global attention provides benefit over CNN for white tea buds
# Note: requires more memory — reduce batch if OOM error

print('Training RT-DETR...')

rtdetr = YOLO('rtdetr-l.pt')

rtdetr.train(
    data     = DATA_YAML,
    epochs   = EPOCHS,
    imgsz    = IMG_SIZE,
    batch    = 8,          # lower batch — transformer is heavier
    name     = 'rtdetr_whitetea',
    project  = PROJECT_DIR,
    patience = PATIENCE,
    save     = True,
    plots    = True,
    exist_ok = True
)

rt_metrics = rtdetr.val(data=DATA_YAML)
print(f'RT-DETR  mAP50: {rt_metrics.box.map50:.4f}  mAP50-95: {rt_metrics.box.map:.4f}')

Training RT-DETR...
Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/augment_herman/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=rtdetr_whitetea, nbs=64, nms=False, opset=None, optimize=False, optimizer=aut

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/60      6.62G      2.388     0.5464     0.7998         17        640: 100% ━━━━━━━━━━━━ 43/43 1.9s/it 1:22
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.6s/it 2.6s
                   all          7         28   0.000952     0.0714   0.000595   8.43e-05

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/60      6.64G      2.696     0.1281     0.8853         65        640: 0% ──────────── 0/43  0.7s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/60      6.64G      2.269    0.09178     0.5245         45        640: 100% ━━━━━━━━━━━━ 43/43 1.3s/it 56.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 3.9it/s 0.3s
                   all          7         28          0          0          0          0

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/60      6.69G      1.883     0.1724     0.4546         58        640: 0% ──────────── 0/43  0.7s

...

      27/60      6.69G      2.072     0.1823     0.3421         45        640: 100% ━━━━━━━━━━━━ 43/43 1.4s/it 1:02
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 4.0it/s 0.3s
                   all          7         28   0.000524     0.0357   0.000375   7.49e-05

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      28/60      6.69G      1.957     0.1815      0.295         67        640: 0% ──────────── 0/43  0.8s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      28/60      6.69G      1.972     0.2137     0.3068         61        640: 100% ━━━━━━━━━━━━ 43/43 1.4s/it 1:02
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 4.0it/s 0.2s
                   all          7         28   0.000559     0.0357   0.000364   0.000109

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      29/60      6.69G      2.388     0.1146     0.4791         61        640: 0% ──────────── 0/43  0.8s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      29/60      6.69G      2.053     0.1911     0.3406         80        640: 100% ━━━━━━━━━━━━ 43/43 1.4s/it 1:00
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 3.7it/s 0.3s
                   all          7         28    0.00056     0.0357   0.000375   0.000112
EarlyStopping: Training stopped early as no improvement observed in last 20 epochs. Best results observed at epoch 9, best model saved as best.pt.
To update EarlyStopping(patience=20) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

29 epochs completed in 0.571 hours.
Optimizer stripped from /content/drive/MyDrive/yolo_runs/rtdetr_whitetea/weights/last.pt, 66.2MB
Optimizer stripped from /content/drive/MyDrive/yolo_runs/rtdetr_whitetea/weights/best.pt, 66.2MB

Validating /content/drive/MyDrive/yolo_runs/rtdetr_whitetea/weights/best.pt...
Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Faster R-CNN

In [11]:
!pip install torchvision -q

In [12]:
# Faster R-CNN + FPN — two-stage quality ceiling
# Ren, S. et al. (2016) ‘Faster R-CNN: Towards real-time object detection with region proposal networks’
# Lin, T.-Y. et al. (2017) ‘Feature Pyramid Networks for Object Detection’
# Establishes upper-bound accuracy reference for one-stage vs two-stage comparison

import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torch

# Load pretrained model
frcnn = fasterrcnn_resnet50_fpn_v2(weights='DEFAULT')

# Replace head for 2 classes (background + unopened_leaf)
in_features = frcnn.roi_heads.box_predictor.cls_score.in_features
frcnn.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes=2)

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
frcnn.to(device)

print('Faster R-CNN model ready.')
print(f'Running on: {device}')
print('NOTE: Faster R-CNN needs a separate dataloader — see next cell.')

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth


100%|██████████| 167M/167M [00:02<00:00, 62.0MB/s]


Faster R-CNN model ready.
Running on: cuda
NOTE: Faster R-CNN needs a separate dataloader — see next cell.


Faster R-CNN dataloader and training loop

In [14]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
import os, cv2
import numpy as np

class WhiteTeaDataset(Dataset):
    def __init__(self, img_dir, label_dir, transforms=None):
        self.img_dir   = img_dir
        self.label_dir = label_dir
        self.transforms = transforms
        self.imgs = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_file   = self.imgs[idx]
        img_path   = os.path.join(self.img_dir, img_file)
        label_path = os.path.join(self.label_dir, img_file.replace('.jpg', '.txt'))

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        boxes, labels = [], []
        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        cls, cx, cy, bw, bh = map(float, parts)
                        # Convert YOLO normalised → absolute xyxy
                        x1 = (cx - bw/2) * w
                        y1 = (cy - bh/2) * h
                        x2 = (cx + bw/2) * w
                        y2 = (cy + bh/2) * h
                        boxes.append([x1, y1, x2, y2])
                        labels.append(1)  # class 1 (0 = background)

        target = {
            'boxes':  torch.tensor(boxes,  dtype=torch.float32) if boxes else torch.zeros((0,4), dtype=torch.float32),
            'labels': torch.tensor(labels, dtype=torch.int64)   if labels else torch.zeros((0,),  dtype=torch.int64),
        }

        img_tensor = T.ToTensor()(img)
        return img_tensor, target


# Dataloaders
train_ds = WhiteTeaDataset(
    f'{PROJECT_DIR}/../augment_herman/train/images',
    f'{PROJECT_DIR}/../augment_herman/train/labels'
)
val_ds = WhiteTeaDataset(
    f'{PROJECT_DIR}/../augment_herman/valid/images',
    f'{PROJECT_DIR}/../augment_herman/valid/labels'
)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,
                          collate_fn=lambda x: tuple(zip(*x)))
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False,
                          collate_fn=lambda x: tuple(zip(*x)))

# Optimiser
optimizer = torch.optim.SGD(frcnn.parameters(), lr=0.005,
                             momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

# Training loop
NUM_EPOCHS = 45
frcnn.train()

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0
    for imgs, targets in train_loader:
        imgs    = [i.to(device) for i in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss_dict = frcnn(imgs, targets)
        losses = sum(loss_dict.values())
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        epoch_loss += losses.item()

    lr_scheduler.step()
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{NUM_EPOCHS}  Loss: {epoch_loss/len(train_loader):.4f}')

# Save weights
torch.save(frcnn.state_dict(),
           f'{PROJECT_DIR}/fasterrcnn_whitetea.pth')
print('Faster R-CNN saved.')

KeyboardInterrupt: 

Save all of them

In [15]:
import shutil, os

# ── Clean weights folder in Drive ───────────────────────────────────────────
WEIGHTS_DIR = '/content/drive/MyDrive/yolo_weights'
os.makedirs(WEIGHTS_DIR, exist_ok=True)

# ── 1. Save YOLO variant weights (best.pt and last.pt) ──────────────────────
for run_name in MODELS.keys():
    best_pt = f'{PROJECT_DIR}/{run_name}/weights/best.pt'
    last_pt = f'{PROJECT_DIR}/{run_name}/weights/last.pt'

    if os.path.exists(best_pt):
        dest = f'{WEIGHTS_DIR}/{run_name}_best.pt'
        shutil.copy2(best_pt, dest)
        print(f'Saved: {dest}')
    else:
        print(f'WARNING: best.pt not found for {run_name}')

    if os.path.exists(last_pt):
        dest = f'{WEIGHTS_DIR}/{run_name}_last.pt'
        shutil.copy2(last_pt, dest)
        print(f'Saved: {dest}')

# ── 2. Save RT-DETR weights ──────────────────────────────────────────────────
rtdetr_best = f'{PROJECT_DIR}/rtdetr_whitetea/weights/best.pt'
rtdetr_last = f'{PROJECT_DIR}/rtdetr_whitetea/weights/last.pt'

if os.path.exists(rtdetr_best):
    shutil.copy2(rtdetr_best, f'{WEIGHTS_DIR}/rtdetr_whitetea_best.pt')
    print(f'Saved: {WEIGHTS_DIR}/rtdetr_whitetea_best.pt')
else:
    print('WARNING: RT-DETR best.pt not found')

if os.path.exists(rtdetr_last):
    shutil.copy2(rtdetr_last, f'{WEIGHTS_DIR}/rtdetr_whitetea_last.pt')
    print(f'Saved: {WEIGHTS_DIR}/rtdetr_whitetea_last.pt')

# ── 3. Save Faster R-CNN weights ─────────────────────────────────────────────
frcnn_path = f'{PROJECT_DIR}/fasterrcnn_whitetea.pth'

if os.path.exists(frcnn_path):
    shutil.copy2(frcnn_path, f'{WEIGHTS_DIR}/fasterrcnn_whitetea.pth')
    print(f'Saved: {WEIGHTS_DIR}/fasterrcnn_whitetea.pth')
else:
    print('WARNING: Faster R-CNN .pth not found')

# ── Summary ───────────────────────────────────────────────────────────────────
print(f'\nAll weights saved to: {WEIGHTS_DIR}')
print('\nContents:')
for f in sorted(os.listdir(WEIGHTS_DIR)):
    size_mb = os.path.getsize(f'{WEIGHTS_DIR}/{f}') / (1024*1024)
    print(f'  {f:<45} {size_mb:.1f} MB')

Saved: /content/drive/MyDrive/yolo_weights/yolov5nu_whitetea_best.pt
Saved: /content/drive/MyDrive/yolo_weights/yolov5nu_whitetea_last.pt
Saved: /content/drive/MyDrive/yolo_weights/yolov8n_whitetea_best.pt
Saved: /content/drive/MyDrive/yolo_weights/yolov8n_whitetea_last.pt
Saved: /content/drive/MyDrive/yolo_weights/yolov10n_whitetea_best.pt
Saved: /content/drive/MyDrive/yolo_weights/yolov10n_whitetea_last.pt
Saved: /content/drive/MyDrive/yolo_weights/yolo11n_whitetea_best.pt
Saved: /content/drive/MyDrive/yolo_weights/yolo11n_whitetea_last.pt
Saved: /content/drive/MyDrive/yolo_weights/rtdetr_whitetea_best.pt
Saved: /content/drive/MyDrive/yolo_weights/rtdetr_whitetea_last.pt

All weights saved to: /content/drive/MyDrive/yolo_weights

Contents:
  rtdetr_whitetea_best.pt                       63.1 MB
  rtdetr_whitetea_last.pt                       63.1 MB
  yolo11n_whitetea_best.pt                      5.2 MB
  yolo11n_whitetea_last.pt                      5.2 MB
  yolov10n_whitetea_best.p